# 03 — JA3 Fingerprint Hunting

**Goal:** enumerate every TLS JA3 we saw, surface rare ones, and drill into known-bad hits.

EDR cannot do this. Only deep observability (Gigamon) sees JA3 fingerprints at scale.

## Setup

Authenticate with Azure CLI (`az login`) and load workspace coordinates from `.env`.

In [ ]:
import os, datetime as dt
import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

load_dotenv()
WORKSPACE_ID = os.environ['WORKSPACE_ID']
HOURS = int(os.environ.get('TIMERANGE_HOURS', '24'))
client = LogsQueryClient(DefaultAzureCredential())
TIMESPAN = dt.timedelta(hours=HOURS)

def kql(q: str) -> pd.DataFrame:
    """Run a KQL query and return the first table as a DataFrame."""
    r = client.query_workspace(WORKSPACE_ID, q, timespan=TIMESPAN)
    if r.status != LogsQueryStatus.SUCCESS:
        raise RuntimeError(r.partial_error)
    t = r.tables[0]
    return pd.DataFrame(t.rows, columns=[c for c in t.columns])

## Pull all JA3s

In [ ]:
ja3 = kql('''
GigamonCcfMcpDemo_CL
| where isnotempty(ssl_fingerprint_ja3)
| summarize Handshakes=count(),
            Sources=dcount(src_ip),
            SampleSnis=make_set(ssl_common_name, 5),
            SampleIssuers=make_set(ssl_certificate_dn_issuer, 5)
  by Ja3=ssl_fingerprint_ja3
| order by Handshakes desc
''')
ja3

## Distribution: how rare is each JA3?

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,5))
ja3_sorted = ja3.sort_values('Handshakes', ascending=False).head(25)
plt.bar(range(len(ja3_sorted)), ja3_sorted['Handshakes'], color='#FF6600')
plt.xticks(range(len(ja3_sorted)), [j[:10] for j in ja3_sorted['Ja3']], rotation=60, ha='right', fontsize=8)
plt.ylabel('Handshakes'); plt.title('Top 25 JA3 fingerprints by volume')
plt.tight_layout(); plt.show()

## Match against the known-bad list

Same list the `Gigamon_JA3_Threat_Match` MCP tool uses — kept inline here for explorability.

In [ ]:
KNOWN_BAD = {
  'e7d705a3286e19ea42f587b344ee6865': 'Trickbot',
  'a0e9f5d64349fb13191bc781f81f42e1': 'Emotet',
  '37f463bf4616ecd445d4a1937da06e19': 'Adwind RAT',
  '72a589da586844d7f0818ce684948eea': 'Tor browser',
  '51c64c77e60f3980eea90869b68c58a8': 'Sliver C2',
  'e5f1a47e3d5d8b3f3e2b4f53a3c6a32f': 'Cobalt Strike',
}
ja3['ThreatLabel'] = ja3['Ja3'].map(KNOWN_BAD)
hits = ja3[ja3['ThreatLabel'].notnull()]
hits

## Drill: who's hitting Cobalt Strike?

In [ ]:
if not hits.empty:
    target = hits.iloc[0]['Ja3']
    detail = kql(f'''
GigamonCcfMcpDemo_CL
| where ssl_fingerprint_ja3 == "{target}"
| project TimeGenerated, src_ip, dst_ip, dst_port, ssl_common_name, ssl_certificate_dn_issuer, total_bytes
| order by TimeGenerated desc
| take 50
''')
    display(detail)
else:
    print('No known-bad JA3 hits in current window.')